In [ ]:
# https://www.kaggle.com/code/jhoward/getting-started-with-nlp-for-absolute-beginners

In [ ]:
# Side note: this nb will use Hugging Face Transformers (Some other work with ULMFit
# was done with RNNs - but Transformers has been the more populat breakthrough)
# ULMFit involved training a language model on Wikipedia data to predict the next word 
# (this didn't work well with Transformers so they used a masked language model which
# would guess blanks inside sentences). This language model was then trained over 
# IMBDB data to predict the next word there, and finally this was used to do a
# classification tasks i.e. sentiment analysis of IMDB reviews

# This is also known as "fine-tuning" i.e. a lot of parameters of a model are pre-trained
# on some other tasks and then we only fine-tune a few especially for our task

In [ ]:
# A widely used NLP application is classification e.e. classifying a document into
# some catgeory. eg:
# Sentiment analysis (e.g are people saying positive or negative things about your product)
# Author identification (what author most likely wrote some document)
# Legal discovery (which documents are in scope for a trial)
# Organizing documents by topic
# Triaging inbound emails

In [ ]:
# Classification models can also be used to solve problems that are not, at first, 
# obviously appropriate. For instance, consider the Kaggle U.S. Patent Phrase to 
# Phrase Matching competition. In this, we are tasked with comparing two words or 
# short phrases, and scoring them based on whether they're similar or not, based on 
# which patent class they were used in. With a score of 1 it is considered that the 
# two inputs have identical meaning, and 0 means they have totally different meaning. 
# For instance, abatement and eliminating process have a score of 0.5, meaning 
# they're somewhat similar, but not identical.

In [ ]:
# Data from https://www.kaggle.com/competitions/us-patent-phrase-to-phrase-matching/data

In [12]:
import pandas as pd

df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [13]:
df.head()

,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


In [14]:
df.describe(include='object')

/tmp/ipykernel_11943/87514550.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,id,anchor,target,context
count,36473,36473,36473,36473
unique,36473,733,29340,106
top,37d61fd2272659b1,component composite coating,composition,H01
freq,1,152,24,2186


In [1]:
# Now we have some documents to do NLP with. But the problem is that neural netowrks
# work with numbers. They take in some numbers, multiple them by matrices, replace the
# negative numbers with zeroes and add them up; and we do this a few time. 
# But how do we do that with these string?
# 

In [15]:
# It turns out that this can be represented as a classification problem. How? By representing the question like this:

# For the following text...: "TEXT1: abatement; TEXT2: eliminating process" ...chose a category of meaning similarity: "Different; Similar; Identical".

In [17]:
df['input'] = 'TEXT1: ' + df.context + '; TEXT2: ' + df.target + '; ANC1: ' + df.anchor

In [18]:
df.input.head() # or df["input"].head()

0    TEXT1: A47; TEXT2: abatement of pollution; ANC...
1    TEXT1: A47; TEXT2: act of abating; ANC1: abate...
2    TEXT1: A47; TEXT2: active catalyst; ANC1: abat...
3    TEXT1: A47; TEXT2: eliminating process; ANC1: ...
4    TEXT1: A47; TEXT2: forest region; ANC1: abatement
Name: input, dtype: str

In [19]:
# Tokenization
# Transformers uses a Dataset object for storing a .... a dataset

from datasets import Dataset,DatasetDict

ds = Dataset.from_pandas(df)

In [20]:
ds

Dataset({
    features: ['id', 'anchor', 'target', 'context', 'score', 'input'],
    num_rows: 36473
})

In [ ]:
# But we can't pass the texts directly into a model. A deep learning model expects 
# numbers as inputs, not English sentences! So we need to do two things:

# 1. Tokenization: Split each text up into words (or actually, as we'll see, into 
# tokens)
# 2. Numericalization: Convert each word (or token) into a number.

# The details about how this is done actually depend on the particular model we use. 
# So first we'll need to pick a model. There are thousands of models available, 
# but a reasonable starting point for nearly any NLP problem is to use this (replace 
# "small" with "large" for a slower but more accurate model, once you've finished 
# exploring):

In [21]:
model_nm = 'microsoft/deberta-v3-small'

In [ ]:
# AutoTokenizer will create a tokenizer appropriate for a given model:

In [23]:
# !pip install tiktoken

/bin/bash: line 1: pip: command not found


In [25]:
from transformers import AutoModelForSequenceClassification,AutoTokenizer

tokz = AutoTokenizer.from_pretrained(model_nm)

[transformers] Could not extract SentencePiece model from /home/vaishnavi-asawale/.cache/huggingface/hub/models--microsoft--deberta-v3-small/snapshots/a36c739020e01763fe789b4b85e2df55d6180012/spm.model using sentencepiece library due to 
SentencePieceExtractor requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: Error parsing line b'\x0e' in /home/vaishnavi-asawale/.cache/huggingface/hub/models--microsoft--deberta-v3-small/snapshots/a36c739020e01763fe789b4b85e2df55d6180012/spm.model